**Install necessary libraries and packages**

In [92]:
!pip install wandb alibi alibi-detect scikit-learn joblib


In [93]:
# Import the NumPy library for numerical operations
import numpy as np
np.__version__

'1.26.4'

In [94]:
import alibi # alibi: Library for explainability and understanding machine learning models
#TabularDrift: Used to detect data drift in numerical features using the Kolmogorov-Smirnov (KS) test
from alibi_detect.cd import ChiSquareDrift, TabularDrift # ChiSquareDrift: Used to detect data drift in categorical features
# save_detector: Used to save a drift detector object to a file
# load_detector: Used to load a saved drift detector object from a file
from alibi_detect.saving import save_detector, load_detector

In [95]:
import pandas as pd # pandas: For data manipulation and analysis
import numpy as np # numpy: For numerical computations and handling arrays
import matplotlib.pyplot as plt # matplotlib: For plotting graphs
import seaborn as sn # seaborn: For statistical data visualization

**Load the real estate dataset from a URL into a Pandas DataFrame**


In [96]:
# Load the real estate dataset from a URL into a Pandas DataFrame
Re_df = pd.read_csv( "https://raw.githubusercontent.com/karapradeepkumar/data/main/Real%20Estate%20Data%20V21.csv" )

In [97]:
# Display a summary of the DataFrame, including column data types, non-null values, and memory usage
Re_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14528 entries, 0 to 14527
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Name            14528 non-null  object 
 1   Property Title  14528 non-null  object 
 2   Price           14528 non-null  object 
 3   Location        14528 non-null  object 
 4   Total_Area      14528 non-null  int64  
 5   Price_per_SQFT  14528 non-null  float64
 6   Description     14528 non-null  object 
 7   Baths           14528 non-null  int64  
 8   Balcony         14528 non-null  object 
dtypes: float64(1), int64(2), object(6)
memory usage: 1021.6+ KB


In [98]:
# Extract and store the list of feature (column) names from the DataFrame
# Re_df.columns returns an Index object containing all column names in the DataFrame
x_features = list(Re_df.columns)

In [99]:
# Display the list of column names extracted from the DataFrame
x_features

['Name',
 'Property Title',
 'Price',
 'Location',
 'Total_Area',
 'Price_per_SQFT',
 'Description',
 'Baths',
 'Balcony']

**Specify the index of the columns which are categorical feautures**

In [100]:
# cat_vars represents the indices of categorical features in the DataFrame.
cat_vars = [0, 1, 2, 3, 6, 8]

In [101]:
# X: Feature matrix containing all columns from the DataFrame specified by x_features
# y: Target vector, in this case, the 'Price' column from the DataFrame
X = Re_df[x_features]
y = Re_df.Price

**Splitting data in training and testing data**

In [102]:
# Import train_test_split from sklearn to split data into training and testing sets
from sklearn.model_selection import train_test_split

In [143]:
# Load the Parquet file from the given URL and save it as X_train and X_prod

# File path for the Parquet file
file_path_1 = 'https://raw.githubusercontent.com/karapradeepkumar/data/main/train_data.parquet'
file_path_2 = 'https://raw.githubusercontent.com/karapradeepkumar/data/main/prod_data.parquet'

# Load the Parquet file into a DataFrame using pandas
X_train = pd.read_parquet(file_path_1)
X_prod = pd.read_parquet(file_path_2)

# Display the first 5 rows of X_train to verify the data has been loaded successfully
print(" X_train Data loaded successfully. Here are the first 5 rows:")
print(X_train.head())

# Print the shape of X_train to confirm the number of rows and columns
print(f"Training feature set (X_train) shape: {X_train.shape}")


 X_train Data loaded successfully. Here are the first 5 rows:
        City              Area_name Property_Type    Brokerage  \
0  Bangalore       Begur, Bangalore          Flat  Not Payable   
1       Pune            Wakad, Pune          Flat  Not Payable   
2       Pune     Keshav Nagar, Pune          Flat  Not Payable   
3  Bangalore  Whitefield, Bangalore          Flat  Not Payable   
4    Kolkata       Taltala, Kolkata          Flat  Not Payable   

                                        Location Baths  Balcony  Total_Area  \
0             Vishwapriya Nagar, Begur,Bangalore     3        0      1260.0   
1  Capital Tower,Shankar Kalat Nagar, Wakad,Pune     3        0      1030.0   
2                             Keshav Nagar, Pune     3        0      1270.0   
3           Prestige Ozone, Whitefield,Bangalore     3        0      1870.0   
4                     Maula Ali, Taltala,Kolkata     3        0      1074.0   

   Bedrooms  Floor  Price_per_SQFT  
0       3.0    3.0          4

**Measure the drift**

In [144]:
 # Create a Tabular Drift detector using Alibi-Detect
    # The reference distribution is set using X_train (the training data)
    # p_val defines the threshold for drift detection (default 0.05)
cd = TabularDrift(X_train.values,
                  p_val=.05,
                  categories_per_feature=categories_per_feature)

"""
    Initializes a Tabular Drift detector to monitor drift in the production data
    when compared to the training data.

      Parameters:
    -----------
    X_train :
        The training feature matrix used as the reference distribution for drift detection.
        This is typically the data that the model was trained on.

    p_val : float, default=0.05
        The significance level (p-value) threshold for detecting drift.
        If the p-value for a feature is less than this threshold,
        the feature is considered to have drifted.

    categories_per_feature : dict, optional, default=None
        A dictionary that defines which features are categorical and their possible categories.
        If None, the detector assumes all features are numerical.

    Returns:
    --------
    cd : TabularDrift object
        A drift detector object that can be used to detect drift in new production data.
    """

'\n   Initializes a Tabular Drift detector to monitor drift in the production data \n   when compared to the training data.\n\n     Parameters:\n   -----------\n   X_train : \n       The training feature matrix used as the reference distribution for drift detection.\n       This is typically the data that the model was trained on.\n   \n   p_val : float, default=0.05\n       The significance level (p-value) threshold for detecting drift. \n       If the p-value for a feature is less than this threshold, \n       the feature is considered to have drifted.\n   \n   categories_per_feature : dict, optional, default=None\n       A dictionary that defines which features are categorical and their possible categories.\n       If None, the detector assumes all features are numerical.\n   \n   Returns:\n   --------\n   cd : TabularDrift object\n       A drift detector object that can be used to detect drift in new production data.\n   '

In [145]:
filepath = 'carsdrift'  # change to directory where detector is saved
save_detector(cd, filepath, legacy = True)

In [146]:
cd = load_detector(filepath)  # Load the saved Tabular Drift detector from the specified file path

In [147]:
# cd.predict() takes the production dataset (X_prod) as input and checks for drift
# X_prod.to_numpy() converts the production DataFrame to a NumPy array since the detector expects a NumPy array
preds = cd.predict(X_prod.to_numpy())

**Printing the test results - KS test for the numerical features and chi-squared test for the categorical features**

In [155]:
# Determine the type of statistical test to display (Chi2 for categorical, K-S for numerical)
# Extract the statistical distance and p-value for the current feature from the prediction results
# Print the feature name, statistical test type, test statistic value, and p-value
for f in range(min(cd.n_features, len(x_features))):  # Iterate up to the minimum length to prevent exceeding index
    stat = 'Chi2' if f in list(categories_per_feature.keys()) else 'K-S'
    fname = x_features[f]
    stat_val, p_val = preds['data']['distance'][f], preds['data']['p_val'][f]
    print(f'{fname} -- {stat} {stat_val:.3f} -- p-value {p_val:.3f}')

"""
    Iterate over each feature in the drift detector to print the test statistics
    and p-values for both categorical and numerical features.

    Parameters:
    -----------
    f : int
        Index of the feature being evaluated.

    Process:
    --------
    - If the feature is categorical (present in categories_per_feature), use the Chi-Square (Chi2) test.
    - If the feature is numerical, use the Kolmogorov-Smirnov (K-S) test.
    - Print the feature name, type of statistical test, test statistic value, and p-value.
  """


Name -- Chi2 13.355 -- p-value 0.064
Property Title -- Chi2 1409.732 -- p-value 0.160
Price -- Chi2 2.047 -- p-value 0.359
Location -- Chi2 0.000 -- p-value 1.000
Total_Area -- K-S 0.028 -- p-value 0.267
Price_per_SQFT -- K-S 0.017 -- p-value 0.846
Description -- Chi2 0.000 -- p-value 1.000
Baths -- K-S 0.028 -- p-value 0.239
Balcony -- Chi2 17.438 -- p-value 0.042


'\n    Iterate over each feature in the drift detector to print the test statistics \n    and p-values for both categorical and numerical features.\n\n    Parameters:\n    -----------\n    f : int\n        Index of the feature being evaluated.\n\n    Process:\n    --------\n    - If the feature is categorical (present in categories_per_feature), use the Chi-Square (Chi2) test.\n    - If the feature is numerical, use the Kolmogorov-Smirnov (K-S) test.\n    - Print the feature name, type of statistical test, test statistic value, and p-value.\n  '

**Checking the distribution of Price_per_SQFT in training and production data**

In [156]:
# Count the occurrences of each unique value in the 'Price_per_SQFT' column of the training dataset (X_train)
# This provides an overview of the frequency distribution of 'Price_per_SQFT' values
X_train.Price_per_SQFT.value_counts()

,count
Price_per_SQFT,
5000.0,84
10000.0,44
6000.0,42
4000.0,39
6670.0,32
...,...
21300.0,1
9610.0,1
1740.0,1


In [157]:
# Count the occurrences of each unique value in the 'Price_per_SQFT' column of the production dataset (X_prod)
# This provides an overview of the frequency distribution of 'Price_per_SQFT' values in the production data
X_prod.Price_per_SQFT.value_counts()

,count
Price_per_SQFT,
5000.0,23
6000.0,14
10000.0,12
4000.0,11
4580.0,11
...,...
20450.0,1
7510.0,1
9050.0,1


**Checking the distribution of Total_Area in training and production data**

In [158]:
# Count the occurrences of each unique value in the 'Total_Area' column of the training dataset (X_train)
# This provides insight into the frequency distribution of different 'Total_Area' values
X_train.Total_Area.value_counts()

,count
Total_Area,
1000.0,131
900.0,122
1200.0,120
1100.0,109
500.0,100
...,...
1178.0,1
806.0,1
1345.0,1


In [159]:
# Count the occurrences of each unique value in the 'Total_Area' column of the production dataset (X_prod)
# This provides insight into the frequency distribution of different 'Total_Area' values in the production data
X_prod.Total_Area.value_counts()

,count
Total_Area,
1000.0,59
1200.0,44
900.0,36
650.0,34
1100.0,34
...,...
1077.0,1
1570.0,1
2625.0,1


**Checking the distribution of Baths in training and production data**

In [160]:
# Count the occurrences of each unique value in the 'Baths' column of the training dataset (X_train)
X_train.Baths.value_counts()

,count
Baths,
3,3079
2,1812
4,162
1,123
5,39
6,20


In [161]:
# Count the occurrences of each unique value in the 'Baths' column of the production dataset (X_prod)
# This shows the frequency distribution of different 'Baths' values (e.g., 1, 2, 3 baths, etc.) in the production data
X_prod.Baths.value_counts()

,count
Baths,
3,1044
2,568
4,64
1,48
5,15
6,7


**Checking the distribution of Balcony in training and production data**

In [162]:
# Count the occurrences of each unique value in the 'Balcony' column of the training dataset (X_train)
X_train.Balcony.value_counts()

,count
Balcony,
0,5235


In [163]:
# Count the occurrences of each unique value in the 'Balcony' column of the production dataset (X_prod)
# This shows the frequency distribution of different 'Balcony' values (e.g., 0, 1, 2 balconies, etc.) in the production data
X_prod.Balcony.value_counts()

,count
Balcony,
0,1746
